# 03 · Fine-tuning de Stable Diffusion 1.5 con LoRA usando 🤗 diffusers (Google Colab con GPU)
**Generación · Especialización en Inteligencia Artificial** — Jahir Ávila

**Requisitos:** Google Colab con GPU (Entorno de ejecución → Cambiar tipo → T4), acceso a Hugging Face. Tiempo aproximado:
20–30 min para 500 pasos a 512 × 512 en una T4.

Flujo (equivalente al del notebook 02, a escala real):
1. Cargar los componentes de Stable Diffusion: **VAE** (imagen ↔ espacio latente 4 × 64 × 64), **codificador de texto CLIP**,
   **U-Net condicionada** y **programador de ruido DDPM**.
2. Congelar todo y agregar adaptadores **LoRA** a la atención de la U-Net (`unet.add_adapter`).
3. Entrenar en el **espacio latente**: `latentes = VAE(imagen)`, `ruido`, `t`, `latentes_ruidosos = add_noise(...)`;
   la U-Net predice el ruido condicionada por el texto; pérdida MSE.
4. Generar imágenes con y sin LoRA y comparar.

Dataset de ejemplo: `lambdalabs/naruto-blip-captions` (1 221 imágenes con descripción). Puede reemplazarse por imágenes propias
(carpeta con `metadata.jsonl` de pares imagen–texto).

In [ ]:
!pip -q install diffusers transformers accelerate peft datasets
import torch, numpy as np, matplotlib.pyplot as plt, time
from datasets import load_dataset
from diffusers import StableDiffusionPipeline, DDPMScheduler
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_diffusers
from torchvision import transforms
assert torch.cuda.is_available(), "Active una GPU: Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU"
dispositivo, tipo = "cuda", torch.float16
MODELO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
PROMPTS = ["a portrait of a man with glasses", "a cat sitting on a chair", "a farmer in a coffee plantation"]

## 1. Componentes de Stable Diffusion y generación antes del ajuste

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(MODELO, torch_dtype=tipo, safety_checker=None).to(dispositivo)
vae, unet, tokenizer, text_encoder = pipe.vae, pipe.unet, pipe.tokenizer, pipe.text_encoder
ruido_sch = DDPMScheduler.from_pretrained(MODELO, subfolder="scheduler")
print(f"U-Net: {sum(p.numel() for p in unet.parameters()) / 1e6:.0f} M parámetros | VAE: {sum(p.numel() for p in vae.parameters()) / 1e6:.0f} M | "
      f"CLIP texto: {sum(p.numel() for p in text_encoder.parameters()) / 1e6:.0f} M")
def generar(prompts, semilla=0, pasos=30):
    g = torch.Generator(dispositivo).manual_seed(semilla)
    return pipe(prompts, num_inference_steps=pasos, guidance_scale=7.5, generator=g).images
antes = generar(PROMPTS)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, im, p in zip(axes, antes, PROMPTS): ax.imshow(im); ax.set_title(p, fontsize=8); ax.axis("off")
plt.suptitle("Antes del fine-tuning"); plt.show()

In [ ]:
# Espacio latente: el VAE comprime 512x512x3 -> 4x64x64 (factor 48)
img = transforms.ToTensor()(antes[0]).unsqueeze(0).to(dispositivo, tipo) * 2 - 1
with torch.no_grad():
    lat = vae.encode(img).latent_dist.sample() * vae.config.scaling_factor
    rec = vae.decode(lat / vae.config.scaling_factor).sample
print("Imagen", tuple(img.shape), "→ latente", tuple(lat.shape))
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(antes[0]); axes[0].set_title("original")
for k in range(4): axes[k + 1].imshow(lat[0, k].float().cpu(), cmap="RdBu_r"); axes[k + 1].set_title(f"latente canal {k}")
axes[5].imshow(((rec[0].float().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu()); axes[5].set_title("reconstrucción VAE")
for a in axes: a.axis("off")
plt.show()

## 2. Datos

In [ ]:
ds = load_dataset("lambdalabs/naruto-blip-captions", split="train")
tf = transforms.Compose([transforms.Resize(512), transforms.CenterCrop(512), transforms.RandomHorizontalFlip(), transforms.ToTensor(),
                         transforms.Normalize([0.5], [0.5])])
def preparar(lote):
    lote["pixel_values"] = [tf(im.convert("RGB")) for im in lote["image"]]
    lote["input_ids"] = tokenizer(lote["text"], padding="max_length", truncation=True, max_length=tokenizer.model_max_length, return_tensors="pt").input_ids
    return lote
ds.set_transform(preparar)
cargador = torch.utils.data.DataLoader(ds, batch_size=2, shuffle=True,
    collate_fn=lambda b: {"pixel_values": torch.stack([x["pixel_values"] for x in b]), "input_ids": torch.stack([x["input_ids"] for x in b])})
print(len(ds), "pares imagen–texto | ejemplo:", ds[0]["text"])

## 3. LoRA sobre la U-Net y entrenamiento en el espacio latente

In [ ]:
unet.requires_grad_(False); vae.requires_grad_(False); text_encoder.requires_grad_(False)
unet.add_adapter(LoraConfig(r=8, lora_alpha=8, init_lora_weights="gaussian", target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
for p in unet.parameters():
    if p.requires_grad: p.data = p.data.float()            # adaptadores en fp32 (entrenamiento mixto)
params = [p for p in unet.parameters() if p.requires_grad]
print(f"Parámetros LoRA entrenables: {sum(p.numel() for p in params) / 1e6:.2f} M "
      f"({100 * sum(p.numel() for p in params) / sum(p.numel() for p in unet.parameters()):.2f} % de la U-Net)")
opt = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-2)
PASOS = 500
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, PASOS)
unet.train(); hist = []; paso = 0; t0 = time.time()
while paso < PASOS:
    for lote in cargador:
        with torch.no_grad():
            lat = vae.encode(lote["pixel_values"].to(dispositivo, tipo)).latent_dist.sample() * vae.config.scaling_factor
            cond = text_encoder(lote["input_ids"].to(dispositivo))[0]
        ruido = torch.randn_like(lat)
        t = torch.randint(0, ruido_sch.config.num_train_timesteps, (lat.size(0),), device=dispositivo)
        lat_ruido = ruido_sch.add_noise(lat, ruido, t)
        with torch.autocast("cuda", dtype=tipo):
            pred = unet(lat_ruido, t, encoder_hidden_states=cond).sample
        perdida = torch.nn.functional.mse_loss(pred.float(), ruido.float())
        opt.zero_grad(); perdida.backward(); torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()
        hist.append(perdida.item()); paso += 1
        if paso % 50 == 0: print(f"Paso {paso} | MSE {np.mean(hist[-50:]):.4f} | {time.time() - t0:.0f} s")
        if paso >= PASOS: break
plt.plot(np.convolve(hist, np.ones(20) / 20, "valid")); plt.title("MSE (media móvil 20)"); plt.xlabel("Paso"); plt.show()

## 4. Generación después del ajuste y guardado de los pesos LoRA

In [ ]:
unet.eval()
despues = generar(PROMPTS)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for k, p in enumerate(PROMPTS):
    axes[0, k].imshow(antes[k]); axes[0, k].set_title("antes: " + p, fontsize=8)
    axes[1, k].imshow(despues[k]); axes[1, k].set_title("después (LoRA): " + p, fontsize=8)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig("sd_antes_despues.png", dpi=120); plt.show()
estado = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
StableDiffusionPipeline.save_lora_weights("lora_sd15", unet_lora_layers=estado)
print("Pesos LoRA guardados en lora_sd15/ (unos pocos MB, frente a ≈ 3,4 GB del modelo completo)")

**Qué observar.** Con el mismo prompt y la misma semilla, las imágenes adoptan el estilo del dataset de ajuste. Los pesos LoRA
pesan unos pocos MB y se cargan sobre el modelo base con `pipe.load_lora_weights("lora_sd15")`. Más pasos o un rango mayor
refuerzan el estilo a costa de perder diversidad (sobreajuste).